In [ ]:
"""
test_sphericity_wadell.ipynb
Author: Henry Yuan
Description: Figure 2 of paper. Tests of sphericity and Wadell roundness
Must run generate_random_synthetics.ipynb before this to generate sphericity data
"""

# imports
import sys
sys.path.append('../../src')

import numpy as np
import matplotlib.pyplot as plt
import synthetic_grains as sg
from format_plots import format_plots
import roundness_3d as r3d
import pandas as pd

# format plots and collect colors
CR, CS, CE, CL, CRD, CED, CWAD, CSTD = format_plots()

# set data directory
data_dir = '../../data/synthetic/'

In [ ]:
# this cell generates Wadell data for grain with aspect 2 of different vertex counts
# saves vertex counts and wadell values

# set save directory
wadell_data_dir = data_dir+'wadell_data_fig2/'

# set aspect ratio
aspect = 2

# make arrays to store data
target_nvs = np.arange(500,10500,500,dtype=int)
nvs_euhedral = np.zeros_like(target_nvs)
wadells_euhedral = np.zeros(len(target_nvs))
nvs_round = np.zeros_like(target_nvs)
wadells_round = np.zeros(len(target_nvs))

# loop through, make grain, and calculate wadell roundness
for i,tnv in enumerate(target_nvs):
    # euhedral
    bv, mesh, nv = sg.synthetic_zircon(tnv, aspect, 0, rotate=False, MIN_VERTS=0,MAX_VERTS=10500)
    nvs_euhedral[i] = nv
    wadells_euhedral[i] = r3d.wadell_curvature_estimate(bv)

    # round
    bv, mesh, nv = sg.synthetic_zircon(tnv, aspect, 1, rotate=False, MIN_VERTS=0,MAX_VERTS=10500)
    nvs_round[i] = nv
    wadells_round[i] = r3d.wadell_curvature_estimate(bv)

# save data
np.save(wadell_data_dir+'euhedral_nvs.npy', nvs_euhedral)
np.save(wadell_data_dir+'round_nvs.npy', nvs_round)
np.save(wadell_data_dir+'euhedral_wadells.npy', wadells_euhedral)
np.save(wadell_data_dir+'round_wadells.npy', wadells_round)

In [ ]:
# collect sphericity, wadell data

# sphericity data just comes from crofton sphericity in normal synthetic roundness data
# 1000 random euhedral, 1000 random round synthetic grains
sphericity_data_file = data_dir + 'random_synthetics/synthetic_stats.csv'

synthetic_stats_df = pd.read_csv(sphericity_data_file)

nvs = synthetic_stats_df['vertex_count']
sphericity = synthetic_stats_df['sphericity_crofton']

# separate into sphericity of euhedral
euhedral_sph = sphericity[synthetic_stats_df['roundness'] == 0]
round_sph = sphericity[synthetic_stats_df['roundness'] == 1]

# wadell comes from data saved in previous cell
nvs_euhedral = np.load(wadell_data_dir+'euhedral_nvs.npy')
nvs_round = np.load(wadell_data_dir+'round_nvs.npy')
wadells_euhedral = np.load(wadell_data_dir+'euhedral_wadells.npy')
wadells_round = np.load(wadell_data_dir+'round_wadells.npy')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3))
axs = axs.flatten()

# sphericity in first subplot
# make bins
sph_max = np.maximum(euhedral_sph.max(), round_sph.max())
sph_min = np.minimum(euhedral_sph.min(), round_sph.min())
print(sph_max)
print(sph_min)
lo = 0.75 # np.floor(sph_min*10) / 10 # manually set
hi = np.ceil(sph_max*10) / 10
bin_size=0.025
bins = np.arange(lo, hi+bin_size,bin_size)
axs[0].hist([euhedral_sph,round_sph],bins=bins,histtype='barstacked',color=[CE,CR],edgecolor=CL, linewidth=0.5,label=['euhedral','round'])
axs[0].legend()
axs[0].set_xlabel(r'$\psi$')
axs[0].set_ylabel('Frequency')

# Wadell roundness in last subplot
axs[1].plot(nvs_round, wadells_round, marker='o', markersize=5, lw=1.5, color=CR, label='round')
axs[1].plot(nvs_euhedral, wadells_euhedral, marker='s', markersize=5, lw=1.5, color=CE,label='euhedral')
axs[1].axvline(5000,ls='--',lw=1,color='black')
axs[1].axvline(1000,ls='--',lw=1,color='black')
axs[1].legend()
axs[1].set_xlabel('Vertex count')
axs[1].set_ylabel(r'$R_{W,\kappa}$')
axs[1].set_xlim([0, 10500])
axs[1].set_ylim([0.3, 0.9])

plt.tight_layout()
plt.savefig('fig2.pdf')
